<a href="https://colab.research.google.com/github/MalakSalehh/Thesis-datasets/blob/main/FINAL_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai pandas numpy scipy scikit-learn

In [ ]:
import re
import json
import time
import numpy as np
import pandas as pd
from openai import OpenAI

In [ ]:
GROQ_API_KEY = "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"

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
url = "https://raw.githubusercontent.com/MalakSalehh/Thesis-datasets/main/training_set_rel3.tsv"
df = pd.read_csv(url, sep="\t", encoding="latin1")
print(df.shape)
df.head()

In [ ]:
set1 = df[df["essay_set"] == 1].copy()
set1 = set1[["essay_id", "essay", "domain1_score"]]

set1["essay"] = set1["essay"].str.replace("\n", " ", regex=False)
set1["essay"] = set1["essay"].str.strip()

print(set1.shape)
set1.head()

In [ ]:
def categorize(score):
    if score <= 5:
        return "Low"
    elif score <= 8:
        return "Medium"
    else:
        return "High"

set1["score_category"] = set1["domain1_score"].apply(categorize)
set1["domain1_score"].value_counts().sort_index()

In [ ]:
cal_low = set1[set1["score_category"] == "Low"].sample(2, random_state=42)
cal_med = set1[set1["score_category"] == "Medium"].sample(2, random_state=42)
cal_high = set1[set1["score_category"] == "High"].sample(2, random_state=42)

calibration = pd.concat([cal_low, cal_med, cal_high]).sample(frac=1, random_state=42).reset_index(drop=True)
calibration

In [ ]:
calibration_ids = calibration["essay_id"].tolist()
demo = set1[~set1["essay_id"].isin(calibration_ids)].sample(30, random_state=42).reset_index(drop=True)
demo

In [ ]:
def format_calibration_examples(calibration_df):
    examples = ""
    for i, row in calibration_df.iterrows():
        examples += f"""
Calibration Example {i+1}

Essay:
{row['essay']}

Human Score: {row['domain1_score']}
Category: {row['score_category']}
"""
    return examples

calibration_text = format_calibration_examples(calibration)

In [ ]:
rubric_text = """
Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores.
Final score range: 2–12.
"""

In [ ]:
def build_baseline_prompt(essay_text):
    return f"""
You are an expert essay grader evaluating student essays.

{rubric_text}

Calibration Examples:
{calibration_text}

Now evaluate the following essay.

Essay:
{essay_text}

Instructions:
1. Predict the final score between 2 and 12.
2. Predict the category: Low, Medium, or High.
3. Explain your reasoning.

Important:
Base your scoring strictly on the rubric and calibration examples.

Return your answer in this format:

Final Score:
Category:
Reasoning:
"""

In [ ]:
def get_response(prompt):

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",  # Working as of April 2026
        messages=[
            {"role": "system", "content": "You are an expert essay grader. Follow the rubric strictly."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,  # Lower temperature for more consistent scoring
        max_tokens=4096
    )
    return response.choices[0].message.content

# Re-run your 5 essays
results = []

for i, row in demo.iterrows():
    essay_id = row["essay_id"]
    essay_text = row["essay"]
    human_score = row["domain1_score"]

    prompt = build_baseline_prompt(essay_text)
    output = get_response(prompt)

    results.append({
        "essay_id": essay_id,
        "human_score": human_score,
        "model_output": output
    })

    print(f"Done Essay {essay_id}")
    time.sleep(1)

In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:
import re

def extract_score(text):
    if pd.isna(text):
        return None

    match = re.search(r"\*{0,2}Final Score:\*{0,2}\s*(\d+)", str(text), re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

def extract_category(text):
    if pd.isna(text):
        return None

    match = re.search(r"\*{0,2}Category:\*{0,2}\s*(Low|Medium|High)", str(text), re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return None

results_df["predicted_score"] = results_df["model_output"].apply(extract_score)
results_df["predicted_category"] = results_df["model_output"].apply(extract_category)

results_df[["essay_id", "human_score", "model_output", "predicted_score", "predicted_category"]]

In [ ]:
clean_df = results_df.dropna(subset=["predicted_score"]).copy()

from sklearn.metrics import cohen_kappa_score, mean_absolute_error
from scipy.stats import pearsonr

qwk = cohen_kappa_score(clean_df["human_score"], clean_df["predicted_score"], weights="quadratic")
pcc, _ = pearsonr(clean_df["human_score"], clean_df["predicted_score"])
mae = mean_absolute_error(clean_df["human_score"], clean_df["predicted_score"])

print("QWK:", qwk)
print("PCC:", pcc)
print("MAE:", mae)

In [ ]:
from sklearn.metrics import cohen_kappa_score, mean_absolute_error
from scipy.stats import pearsonr

qwk = cohen_kappa_score(
    clean_df["human_score"],
    clean_df["predicted_score"],
    weights="quadratic"
)

pcc, _ = pearsonr(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

mae = mean_absolute_error(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

print("QWK:", qwk)
print("PCC:", pcc)
print("MAE:", mae)

# **IGNORE THE ABOVE , WE ARE TRYING FINE TUNING**

In [ ]:
import pandas as pd

# Load dataset
url = "https://raw.githubusercontent.com/MalakSalehh/Thesis-datasets/main/training_set_rel3.tsv"
df = pd.read_csv(url, sep="\t", encoding="latin1")
print(df.shape)
df.head()

# Keep only essay set 1
set1_df = df[df["essay_set"] == 1].copy()

# Keep only important columns
set1_df = set1_df[["essay_id", "essay_set", "essay", "domain1_score"]].copy()

# Clean essay text
set1_df["essay"] = (
    set1_df["essay"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.strip()
)

# Remove empty essays if any
set1_df = set1_df[set1_df["essay"].str.len() > 0].copy()

print(set1_df.shape)
set1_df.head()

In [ ]:
def score_to_category(score):
    if 2 <= score <= 5:
        return "Low"
    elif 6 <= score <= 8:
        return "Medium"
    elif 9 <= score <= 12:
        return "High"
    else:
        return "Unknown"

set1_df["category"] = set1_df["domain1_score"].apply(score_to_category)

set1_df[["essay_id", "domain1_score", "category"]].head()

In [ ]:
essay_prompt_text = """
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.
""".strip()

rubric_text = """
The essay should be evaluated based on persuasive writing quality.
Score the response according to how clearly it presents a position, how well ideas are developed, how effectively details support the argument, and how organized and coherent the essay is.

Scoring interpretation:
1 = very weak response with minimal development
2 = limited response with weak support
3 = basic response with some development
4 = adequate response with reasonable organization and support
5 = strong response with clear development and effective support
6 = very strong persuasive response with clear organization, strong development, and effective language use

Final score rule:
Two human raters each assign a score from 1 to 6.
The final score is the sum of both ratings, so the final score ranges from 2 to 12.

Category mapping:
2-5 = Low
6-8 = Medium
9-12 = High
""".strip()

print(essay_prompt_text)
print()
print(rubric_text)

In [ ]:
def build_input_text(essay_text):
    return f"""You are an expert essay evaluator.

Evaluate the following student essay using the provided essay prompt and rubric.

Essay Prompt:
{essay_prompt_text}

Rubric:
{rubric_text}

Student Essay:
{essay_text}
""".strip()

def build_output_text(score, category):
    return f"""Score: {score}
Category: {category}
Reasoning: This essay was assigned a {category.lower()} score based on its overall alignment with the rubric criteria for content, support, and organization."""

In [ ]:
set1_df["input_text"] = set1_df["essay"].apply(build_input_text)
set1_df["output_text"] = set1_df.apply(
    lambda row: build_output_text(row["domain1_score"], row["category"]),
    axis=1
)

set1_df[["essay_id", "input_text", "output_text"]].head(1)

In [ ]:
##split the data into train and test
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    set1_df[["essay_id", "input_text", "output_text", "domain1_score", "category"]],
    test_size=0.2,
    random_state=42,
    stratify=set1_df["category"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

  ## - **convert to Hugging Face dataset format**



In [ ]:
!pip install -q datasets
!pip uninstall -y pyarrow datasets
!pip install pyarrow==15.0.2 datasets==2.19.1

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["input_text", "output_text"]])
test_dataset = Dataset.from_pandas(test_df[["input_text", "output_text"]])

print(train_dataset)
print(test_dataset)

- **Change the columns into instruction-tuning format**

      - Most fine-tuning setups want one combined text field

In [ ]:
def format_example(example):
    return {
        "text": f"""### Instruction:
{example['input_text']}

### Response:
{example['output_text']}"""
    }

train_dataset = train_dataset.map(format_example)
test_dataset = test_dataset.map(format_example)

print(train_dataset[0]["text"])

In [ ]:
## install the training libraries
!pip install -q transformers peft accelerate bitsandbytes trl

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def tokenize_function(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_train = train_dataset.map(tokenize_function, batched=False)
tokenized_test = test_dataset.map(tokenize_function, batched=False)

In [ ]:
tokenized_train = tokenized_train.remove_columns(
    [col for col in tokenized_train.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)

tokenized_test = tokenized_test.remove_columns(
    [col for col in tokenized_test.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)

print(tokenized_train)
print(tokenized_test)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="/content/fine_tuned_phi3_aes",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    fp16=True,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
save_path = "/content/drive/MyDrive/phi3_aes_final"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

In [ ]:
model.save_pretrained("/content/fine_tuned_phi3_aes_final")
tokenizer.save_pretrained("/content/fine_tuned_phi3_aes_final")

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
sample_prompt = f"""### Instruction:
{test_df.loc[0, "input_text"]}

### Response:
"""

In [ ]:
result = generator(
    sample_prompt,
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False
)[0]["generated_text"]

print(result)

In [ ]:
sample_prompt = f"""### Instruction:
{test_df.loc[0, "input_text"]}

### Response:
"""

inputs = tokenizer(sample_prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
response_only = result.split("### Response:")[-1].strip()

print(response_only)

In [31]:

import re

def extract_score(text):
    match = re.search(r"Score:\s*(\d+)", text)
    return int(match.group(1)) if match else None

def extract_category(text):
    match = re.search(r"Category:\s*(Low|Medium|High)", text, re.IGNORECASE)
    return match.group(1).capitalize() if match else None

print("Predicted score:", extract_score(response_only))
print("Predicted category:", extract_category(response_only))

Predicted score: None
Predicted category: None


In [ ]:
import json

path = "/content/FINAL.ipynb"   # change if needed

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# remove broken widgets
nb.get("metadata", {}).pop("widgets", None)

for cell in nb.get("cells", []):
    if "metadata" in cell and "widgets" in cell["metadata"]:
        del cell["metadata"]["widgets"]

with open(path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("✅ FIXED")

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".ipynb"):
            print(os.path.join(root, file))